# Какие запросы «похожи на английские»

Два уровня строгости:

1. Только латиница (ASCII)
2. **Язык по библиотеке** `langdetect` 


In [1]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

In [ ]:
OUT_DIR = Path(r"D:\YandexDisk\Гузель\магистратура\Диплом\processed_parquet")
KEYWORD_SOURCE = OUT_DIR / "all_word_metrics.parquet"

BATCH_ROWS = 500_000
USE_KEEP_ONLY = True

ASCII_LATIN_KEYWORD = re.compile(
    r"^[a-zA-Z][a-zA-Z0-9\s'\-\.&]*$"
)

def is_ascii_latin_only(s: str) -> bool:
    s = str(s).strip()
    if len(s) < 2:
        return False
    return bool(ASCII_LATIN_KEYWORD.match(s))

In [3]:
# --- Сбор уникальных keyword + разделение на «латиница» / «остальное» ---
if not KEYWORD_SOURCE.is_file():
    raise FileNotFoundError(KEYWORD_SOURCE)

pf = pq.ParquetFile(KEYWORD_SOURCE)
names = {f.name for f in pf.schema_arrow}
cols = ["keyword"]
if USE_KEEP_ONLY and "keep" in names:
    cols.append("keep")
    filter_keep = True
else:
    filter_keep = False

seen: set[str] = set()
latin_ok: set[str] = set()
latin_fail: set[str] = set()

for batch in pf.iter_batches(batch_size=BATCH_ROWS, columns=cols):
    t = batch.to_pandas()
    if filter_keep:
        t = t.loc[t["keep"]]
    for k in t["keyword"].astype(str).str.strip():
        if not k or k in seen:
            continue
        seen.add(k)
        if is_ascii_latin_only(k):
            latin_ok.add(k)
        else:
            latin_fail.add(k)

print("Фильтр keep=True:", filter_keep)
print("Уникальных keyword:", len(seen))
print("Только ASCII-латиница (по правилу выше):", len(latin_ok))
print("Остальные (кириллица, смешанные, акценты, символы):", len(latin_fail))

Фильтр keep=True: True
Уникальных keyword: 13243045
Только ASCII-латиница (по правилу выше): 13243021
Остальные (кириллица, смешанные, акценты, символы): 24


In [ ]:
import random

rng = random.Random(42)
sample_latin = sorted(latin_ok)[:200] 
sample_latin_r = rng.sample(sorted(latin_ok), min(30, len(latin_ok))) if latin_ok else []
sample_other = rng.sample(sorted(latin_fail), min(30, len(latin_fail))) if latin_fail else []

print("--- примеры «латиница по regex» (случайные 30) ---")
print(", ".join(sample_latin_r))
print("\n--- примеры НЕ прошедших regex (30) ---")
print(", ".join(sample_other))

--- примеры «латиница по regex» (случайные 30) ---
slton, car dar, alumnow, vijuve, ghetto cooking, finest choice, escondidos, coca light, vena cure, butt stop, tabletop now, videopc, poixa, bob sanders, rleeds, machine skills, animation portfolio workshop, and stl, branches wholesale, enmouvement, eze clean, olevian, safi global, ambiente sedona, pseng, dusly, ttmx, spacr, tlcoffer, podsticaji

--- примеры НЕ прошедших regex (30) ---
p, i, q, u, j, a, g, z, l, e, c, d, t, b, r, h, k, s, v, x, m, w, y, n


In [7]:
%pip install langdetect

Using legacy 'setup.py install' for langdetect, since package 'wheel' is not installed.
    Running setup.py install for langdetect: started
    Running setup.py install for langdetect: finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Arthur\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:
MAX_CHECK = 50_000  

LANGDETECT_MIN_LEN = 10

def ok_for_langdetect(s: str) -> bool:
    s = str(s).strip()
    if len(s) < LANGDETECT_MIN_LEN:
        return False
    parts = s.split()
    if not parts:
        return False
    if all(len(t) == 1 and t.isalpha() for t in parts):
        return False
    return True


try:
    from langdetect import detect, LangDetectException, DetectorFactory

    DetectorFactory.seed = 0
except ImportError:
    print("Установи: pip install langdetect")
    raise

words = sorted(latin_ok)
if MAX_CHECK is not None:
    words = words[:MAX_CHECK]

en, not_en, skipped = [], [], []
for w in words:
    if not ok_for_langdetect(w):
        skipped.append(w)
        continue
    try:
        (en if detect(w) == "en" else not_en).append(w)
    except LangDetectException:
        not_en.append(w)

checked = len(words)
actually = len(en) + len(not_en)
print(f"Всего в списке (MAX_CHECK): {checked}")
print(f"Пропущено как шум (короткие / только однобуквенные токены): {len(skipped)}")
print(f"Реально вызван langdetect: {actually}")
print("langdetect == en:", len(en))
print("иначе / ошибка:", len(not_en))
print("примеры пропущенного шума:", ", ".join(skipped[:20]))
print("примеры en:", ", ".join(en[:25]))
print("примеры не-en:", ", ".join(not_en[:25]))

Проверено слов: 50000
langdetect == en: 6999
иначе / ошибка: 43001
примеры en: a a aesthetics, a a aparty, a a auctioneers, a a collective, a a f f, a a fashion, a a freight, a a group, a a inc, a a insurance services, a a international, a a intl, a a online, a a photography, a a productions, a a referencement, a a refo, a a ron warren, a a s h, a a service, a a services, a a shoe fetish, a a smith, a a sound, a aberrations
примеры не-en: a a, a a a, a a a a, a a a a a, a a a a a a, a a a a a a a a a a a a a a a a a a a a a a a a a a a, a a a ah, a a a d e, a a a events, a a a g, a a a japan, a a a market, a a a party, a a a s, a a a storage, a a a transportation, a a aaa, a a academy, a a access, a a acupuncture, a a ah, a a ahole, a a air, a a apartments, a a arquitectura
